In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load the final master dataset

file_path = r"Output\Marketing_Campaign_Feature_Engineered_Master.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset loaded successfully!
Shape: (166665, 25)

Columns:
['Campaign_ID', 'Campaign_Type', 'Target_Audience', 'Duration', 'Channel_Used', 'Impressions', 'Clicks', 'Leads', 'Conversions', 'Revenue', 'Acquisition_Cost', 'Language', 'Engagement_Score', 'Customer_Segment', 'Date', 'Company_Name', 'Profit', 'Profit_Flag', 'Calculated_ROI', 'Channel_Google', 'Channel_WhatsApp', 'Channel_YouTube', 'Channel_Email', 'Channel_Instagram', 'Channel_Facebook']


In [3]:
# Validate the classification target

print("Profit_Flag value counts:")
print(df["Profit_Flag"].value_counts())

print("\nProfit_Flag percentage distribution:")
print((df["Profit_Flag"].value_counts(normalize=True) * 100).round(2))

print("\nMissing Profit_Flag values:")
print(df["Profit_Flag"].isna().sum())

Profit_Flag value counts:
Profit_Flag
Profit    158350
Loss        8315
Name: count, dtype: int64

Profit_Flag percentage distribution:
Profit_Flag
Profit    95.01
Loss       4.99
Name: proportion, dtype: float64

Missing Profit_Flag values:
0


In [4]:
# Create binary classification target

y = df["Profit_Flag"].map({
    "Loss": 0,
    "Profit": 1
})

print("Target created successfully!")
print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget percentages:")
print((y.value_counts(normalize=True) * 100).round(2))

Target created successfully!

Target distribution:
Profit_Flag
1    158350
0      8315
Name: count, dtype: int64

Target percentages:
Profit_Flag
1    95.01
0     4.99
Name: proportion, dtype: float64


In [5]:
# Define classification features
# Calculated_ROI is intentionally included for experimental comparison

numerical_features = [
    "Duration",
    "Impressions",
    "Clicks",
    "Leads",
    "Conversions",
    "Acquisition_Cost",
    "Calculated_ROI",
    "Engagement_Score",
    "Channel_Google",
    "Channel_WhatsApp",
    "Channel_YouTube",
    "Channel_Email",
    "Channel_Instagram",
    "Channel_Facebook"
]

categorical_features = [
    "Campaign_Type",
    "Target_Audience",
    "Channel_Used",
    "Language",
    "Customer_Segment",
    "Company_Name"
]

all_features = numerical_features + categorical_features

X = df[all_features].copy()

print("Classification features created successfully!")
print("\nTotal features:", len(all_features))
print("\nNumerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

print("\nX shape:", X.shape)
print("\ny shape:", y.shape)

Classification features created successfully!

Total features: 20

Numerical features: 14
Categorical features: 6

X shape: (166665, 20)

y shape: (166665,)


In [6]:
# Check missing values in classification features

missing_values = X.isnull().sum()

print("Missing values in classification features:")
print(missing_values[missing_values > 0])

print("\nTotal missing values:", X.isnull().sum().sum())

Missing values in classification features:
Series([], dtype: int64)

Total missing values: 0


In [7]:
# Split data into training and testing sets
# Stratification preserves the Profit/Loss class distribution

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train/Test split completed!")

print("\nTraining set:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting set:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTraining target distribution:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting target distribution:")
print((y_test.value_counts(normalize=True) * 100).round(2))

Train/Test split completed!

Training set:
X_train: (133332, 20)
y_train: (133332,)

Testing set:
X_test: (33333, 20)
y_test: (33333,)

Training target distribution:
Profit_Flag
1    95.01
0     4.99
Name: proportion, dtype: float64

Testing target distribution:
Profit_Flag
1    95.01
0     4.99
Name: proportion, dtype: float64


In [8]:
# One-hot encode categorical features

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

X_train_cat = encoder.fit_transform(X_train[categorical_features])
X_test_cat = encoder.transform(X_test[categorical_features])

print("Categorical encoding completed!")

print("\nEncoded training categorical shape:", X_train_cat.shape)
print("Encoded testing categorical shape:", X_test_cat.shape)

Categorical encoding completed!

Encoded training categorical shape: (133332, 178)
Encoded testing categorical shape: (33333, 178)


In [9]:
from scipy.sparse import hstack

# Extract numerical features
X_train_num = X_train[numerical_features].values
X_test_num = X_test[numerical_features].values

# Combine numerical and encoded categorical features
X_train_final = hstack([X_train_num, X_train_cat])
X_test_final = hstack([X_test_num, X_test_cat])

print("Final feature matrices created!")

print("\nX_train_final shape:", X_train_final.shape)
print("X_test_final shape:", X_test_final.shape)

Final feature matrices created!

X_train_final shape: (133332, 192)
X_test_final shape: (33333, 192)


In [10]:
from sklearn.ensemble import GradientBoostingClassifier

# Initialize Gradient Boosting Classifier
gb_classifier = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)

print("Gradient Boosting Classifier initialized!")

Gradient Boosting Classifier initialized!


In [11]:
# Train Gradient Boosting Classifier

print("Training Gradient Boosting Classifier...")

gb_classifier.fit(X_train_final, y_train)

print("Training completed successfully!")

Training Gradient Boosting Classifier...
Training completed successfully!


In [12]:
# Generate predictions using Gradient Boosting Classifier

y_pred_gb = gb_classifier.predict(X_test_final)

print("Predictions generated successfully!")
print("Number of predictions:", len(y_pred_gb))

Predictions generated successfully!
Number of predictions: 33333


In [13]:
# Evaluate Gradient Boosting Classifier

accuracy_gb = accuracy_score(y_test, y_pred_gb)
precision_gb = precision_score(y_test, y_pred_gb)
recall_gb = recall_score(y_test, y_pred_gb)
f1_gb = f1_score(y_test, y_pred_gb)

print("Gradient Boosting Classification Performance")
print("-" * 50)
print(f"Accuracy : {accuracy_gb:.4f}")
print(f"Precision: {precision_gb:.4f}")
print(f"Recall   : {recall_gb:.4f}")
print(f"F1-Score : {f1_gb:.4f}")

Gradient Boosting Classification Performance
--------------------------------------------------
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1-Score : 1.0000


In [14]:
# Detailed classification report for Gradient Boosting

print("Gradient Boosting Classification Report")
print("-" * 55)

print(
    classification_report(
        y_test,
        y_pred_gb,
        target_names=["Loss", "Profit"],
        digits=4
    )
)

Gradient Boosting Classification Report
-------------------------------------------------------
              precision    recall  f1-score   support

        Loss     1.0000    1.0000    1.0000      1663
      Profit     1.0000    1.0000    1.0000     31670

    accuracy                         1.0000     33333
   macro avg     1.0000    1.0000    1.0000     33333
weighted avg     1.0000    1.0000    1.0000     33333



In [15]:
cm_gb = confusion_matrix(y_test, y_pred_gb)

print("Gradient Boosting Confusion Matrix")
print("-" * 45)
print(cm_gb)

Gradient Boosting Confusion Matrix
---------------------------------------------
[[ 1663     0]
 [    0 31670]]


In [16]:
import pickle
import os

# Output folder
output_folder = r"D:\Data Science\vscode\Marketing_Campaign_Performance_Prediction\Output"

# Save the trained classification model
classification_model_path = os.path.join(
    output_folder,
    "classification_gradient_boosting_model.pkl"
)

with open(classification_model_path, "wb") as file:
    pickle.dump(gb_classifier, file)

# Save the categorical encoder
classification_encoder_path = os.path.join(
    output_folder,
    "classification_encoder.pkl"
)

with open(classification_encoder_path, "wb") as file:
    pickle.dump(encoder, file)

print("Classification model and encoder saved successfully!")
print("\nModel:", classification_model_path)
print("Encoder:", classification_encoder_path)

Classification model and encoder saved successfully!

Model: D:\Data Science\vscode\Marketing_Campaign_Performance_Prediction\Output\classification_gradient_boosting_model.pkl
Encoder: D:\Data Science\vscode\Marketing_Campaign_Performance_Prediction\Output\classification_encoder.pkl
